# Medical Domain Adaptation of Llama-3-8B Using QLoRA

**Author:** Alex Wenxuan  
**Date:** February 2026  
**Objective:** Fine-tune Meta's Llama-3-8B model for clinical question-answering using Parameter-Efficient Fine-Tuning (QLoRA) on the Medical Meadow WikiDoc dataset.

---

This notebook provides a complete, reproducible pipeline for:
1. Loading and cleaning medical QA data from Hugging Face
2. Splitting into train/validation sets
3. Fine-tuning Llama-3-8B with multiple LoRA configurations
4. Comparing fine-tuned vs. baseline model performance

**No external files required** — all data is fetched programmatically.

## 1. Environment Setup

Install required packages and authenticate with Hugging Face Hub.

In [ ]:
# Install Unsloth and dependencies
!pip install unsloth-zoo
!pip install --no-deps "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl==0.8.6" peft accelerate bitsandbytes
!pip install -U pyarrow datasets

In [ ]:
# Hugging Face Authentication
from huggingface_hub import login

# For Kaggle: Use Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HuggingFace Token")
    login(hf_token)
    print("✅ Authenticated via Kaggle Secrets")
except:
    # For local/Colab: Manual token entry
    print("⚠️ Kaggle Secrets not available. Please enter your HF token manually:")
    login()

## 2. Data Ingestion & Cleaning

Load the raw Medical Meadow WikiDoc dataset and apply noise filtering:
- Remove empty/short fields
- Filter AI meta-talk ("As an AI language model...")
- Remove low clinical utility responses
- Detect truncated outputs

In [ ]:
import pandas as pd
from datasets import load_dataset
import random
import re

# Set seed for reproducibility
random.seed(42)

def load_and_audit(dataset_name="medalpaca/medical_meadow_wikidoc"):
    """Phase 1: Load and sample dataset for auditing."""
    print(f"Loading dataset: {dataset_name}...")
    dataset = load_dataset(dataset_name, trust_remote_code=True)
    df = pd.DataFrame(dataset['train'])
    
    print(f"Initial row count: {len(df)}")
    
    # Snapshot: 50 random rows
    sample_indices = random.sample(range(len(df)), 50)
    snapshot = df.iloc[sample_indices]
    
    return df, snapshot

def identify_noise(row):
    """Categorize noise patterns for reporting."""
    instruction = str(row['instruction']).strip().lower()
    output = str(row['output']).strip().lower()
    
    # Category 1: Empty/Near-empty
    if len(instruction) < 3 or len(output) < 3:
        return "Empty/Short Field"
    
    # Category 2: Meta-talk
    meta_patterns = [
        "as an ai language model",
        "i'm sorry, but i cannot",
        "the provided text does not contain",
        "the text provided is",
        "as an ai",
        "i do not have access to"
    ]
    if any(pattern in output for pattern in meta_patterns):
        return "Meta-talk"
    
    # Category 3: Clinical Utility (Short output)
    if len(output) < 20:
        return "Low Clinical Utility"
    
    # Category 4: Truncated/Nonsensical
    if len(output) > 0 and output[-1] not in ['.', '?', '!', '"', ')']:
        if len(output) > 100:
            return "Possible Truncation"

    return None

def clean_dataset(df):
    """Phase 2: Apply cleaning logic and track removals."""
    print("\nStarting cleaning process...")
    
    # Initial stats
    initial_count = len(df)
    
    # Identify noise for reporting
    df['noise_category'] = df.apply(identify_noise, axis=1)
    
    # Save a "Before" copy for the examples
    noisy_rows = df[df['noise_category'].notnull()].head(10)
    
    # Filter
    cleaned_df = df[df['noise_category'].isnull()].copy()
    
    # Breakdown statistics
    breakdown = df['noise_category'].value_counts()
    
    print("Cleaning complete.")
    return cleaned_df, breakdown, noisy_rows

# Execute cleaning pipeline
df_raw, snapshot = load_and_audit()
df_cleaned, noise_breakdown, noisy_samples = clean_dataset(df_raw)

# Save cleaned dataset
df_cleaned.drop(columns=['noise_category']).to_json("./cleaned_medical_meadow.jsonl", orient='records', lines=True)

# Report
print("\n" + "="*50)
print("           DATA CLEANING REPORT")
print("="*50)
print(f"Total rows before cleaning: {len(df_raw)}")
print(f"Total rows after cleaning:  {len(df_cleaned)}")
print(f"Rows removed:               {len(df_raw) - len(df_cleaned)}")
print("\nRemoval Breakdown by Category:")
print(noise_breakdown)
print("\n✅ Cleaned dataset saved to: ./cleaned_medical_meadow.jsonl")

## 3. Train/Validation Splitting

Perform a stratified 90/10 split with reproducible seed and analyze sequence statistics.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

def prepare_splits(input_file="./cleaned_medical_meadow.jsonl"):
    """Load cleaned data, perform 90/10 split, and analysis."""
    print(f"Loading cleaned dataset: {input_file}...")
    df = pd.read_json(input_file, orient='records', lines=True)
    
    print(f"Total sanitized rows: {len(df)}")
    
    # 90/10 Split with fixed seed
    train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42)
    
    print(f"\nSplit Results:")
    print(f"Training set:   {len(train_df)} rows")
    print(f"Evaluation set: {len(eval_df)} rows")
    
    # Save splits
    train_df.to_json("./train_split.jsonl", orient='records', lines=True)
    eval_df.to_json("./eval_split.jsonl", orient='records', lines=True)
    print("Files saved: ./train_split.jsonl, ./eval_split.jsonl")
    
    # Analysis: Word count on output field
    print("\nSequence Analysis (Output Field - Training Set):")
    output_word_counts = train_df['output'].apply(lambda x: len(str(x).split()))
    
    stats = {
        "Average Word Count": output_word_counts.mean(),
        "Median Word Count":  output_word_counts.median(),
        "Max Word Count":     output_word_counts.max(),
        "95th Percentile":    output_word_counts.quantile(0.95),
        "99th Percentile":    output_word_counts.quantile(0.99)
    }
    
    for label, val in stats.items():
        print(f"{label:20}: {val:.2f}")
        
    return stats

# Execute splitting
split_stats = prepare_splits()
print("\n✅ Train/Eval splits ready for fine-tuning")

## 4. Model Initialization (Unsloth)

Load the quantized Llama-3-8B model and configure the prompt template.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import gc
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

# Configuration
train_path = "./train_split.jsonl"  # Updated to local path
eval_path = "./eval_split.jsonl"    # Updated to local path

max_seq_length = 512 
dtype = None 
load_in_4bit = True 

# Load Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("✅ Base model loaded successfully")

In [ ]:
# Data Preparation with Prompt Template
prompt_template = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}<|end_of_text|>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        text = prompt_template.format(instruction=inst, input=inp, output=out)
        texts.append(text)
    return { "text" : texts, }

# Load datasets
train_dataset = load_dataset("json", data_files=train_path, split="train")
eval_dataset = load_dataset("json", data_files=eval_path, split="train")

# Apply formatting
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)

print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Evaluation samples: {len(eval_dataset)}")

## 5. Hyperparameter Grid Search

Train 10 different LoRA configurations varying rank, alpha, and learning rate.

In [ ]:
# Define hyperparameter grid
configs = [
    {"rank": 8, "alpha": 16, "lr": 1e-4},
    {"rank": 16, "alpha": 32, "lr": 2e-4},
    {"rank": 16, "alpha": 32, "lr": 1e-4},
    {"rank": 16, "alpha": 32, "lr": 5e-5},
    {"rank": 32, "alpha": 64, "lr": 2e-4},
    {"rank": 32, "alpha": 64, "lr": 1e-4},
    {"rank": 32, "alpha": 64, "lr": 5e-5},
    {"rank": 64, "alpha": 128, "lr": 1e-4},
    {"rank": 64, "alpha": 128, "lr": 5e-5},
]

results = []

for i, config in enumerate(configs):
    print(f"\n{'='*60}")
    print(f"Training Configuration {i+1}/10")
    print(f"Rank: {config['rank']}, Alpha: {config['alpha']}, LR: {config['lr']}")
    print(f"{'='*60}")
    
    # Apply LoRA adapters
    peft_model = FastLanguageModel.get_peft_model(
        model,
        r = config["rank"],
        lora_alpha = config["alpha"],
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 42,
    )

    # Initialize trainer
    trainer = SFTTrainer(
        model = peft_model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = eval_dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        args = TrainingArguments(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            max_steps = 60,
            learning_rate = config["lr"],
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 10,
            eval_strategy = "steps",
            eval_steps = 20,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            output_dir = f"./outputs/config_{i+1}",
            report_to = "none",
        ),
    )

    # Train and evaluate
    trainer.train()
    eval_metrics = trainer.evaluate()
    
    # Extract final training loss
    train_loss = next((log['loss'] for log in reversed(trainer.state.log_history) if 'loss' in log), None)
    
    # Store results
    results.append({
        "Config": i+1,
        "Rank": config["rank"],
        "Alpha": config["alpha"],
        "Learning_Rate": config["lr"],
        "Train_Loss": train_loss,
        "Eval_Loss": eval_metrics['eval_loss']
    })
    
    # Memory cleanup
    del trainer, peft_model
    gc.collect()
    torch.cuda.empty_cache()

# Save grid search results
df_results = pd.DataFrame(results)
df_results.to_csv("./grid_search_results.csv", index=False)

print("\n" + "="*60)
print("GRID SEARCH COMPLETE")
print("="*60)
print(df_results.to_string(index=False))
print("\n✅ Results saved to: ./grid_search_results.csv")

## 6. Evaluation & Inference

### 6.1 Fine-Tuned Model Inference

Test the best-performing configuration on 30 held-out evaluation samples.

In [ ]:
# Identify best configuration based on eval loss
best_config = df_results.loc[df_results['Eval_Loss'].idxmin()]
best_config_num = int(best_config['Config'])

print(f"🏆 Best Configuration: Config {best_config_num}")
print(f"   Rank: {best_config['Rank']}, Alpha: {best_config['Alpha']}, LR: {best_config['Learning_Rate']}")
print(f"   Eval Loss: {best_config['Eval_Loss']:.4f}")

# Path to best adapter
adapter_path = f"./outputs/config_{best_config_num}/checkpoint-60"

# Reload model with best adapter
model_finetuned, tokenizer_finetuned = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Apply adapter and enable inference mode
model_finetuned = FastLanguageModel.for_inference(model_finetuned)
model_finetuned.load_adapter(adapter_path)
tokenizer_finetuned.padding_side = "right"

print(f"✅ Loaded fine-tuned adapter from: {adapter_path}")

In [ ]:
# Select 30 test samples
eval_dataset_full = load_dataset("json", data_files=eval_path, split="train")
test_samples = eval_dataset_full.shuffle(seed=42).select(range(30))

finetuned_results = []
print("🚀 Starting inference on 30 clinical samples (Fine-Tuned Model)...\n")

for i, example in enumerate(test_samples):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n"
    
    inputs = tokenizer_finetuned([prompt], return_tensors = "pt").to("cuda")
    
    outputs = model_finetuned.generate(
        **inputs, 
        max_new_tokens = 150, 
        use_cache = True,
        temperature = 0.1,
        top_p = 0.9
    )
    
    full_response = tokenizer_finetuned.batch_decode(outputs)[0]
    generated_text = full_response.split("### Response:\n")[-1].replace("<|end_of_text|>", "").strip()
    
    finetuned_results.append({
        "ID": i + 1,
        "Instruction": example['instruction'],
        "Clinical_Input": example['input'],
        "Ground_Truth": example['output'],
        "FineTuned_Response": generated_text
    })
    
    if (i + 1) % 5 == 0:
        print(f"✅ Processed {i+1}/30 samples...")

df_finetuned = pd.DataFrame(finetuned_results)
df_finetuned.to_csv("./finetuned_model_evaluation.csv", index=False)
print("\n✅ Fine-tuned model results saved to: ./finetuned_model_evaluation.csv")

### 6.2 Baseline Model Inference

Run the same 30 samples through the **untrained** base model for comparison.

In [ ]:
# Load raw base model (no adapter)
model_base, tokenizer_base = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Enable inference mode
FastLanguageModel.for_inference(model_base)
tokenizer_base.padding_side = "right"

print("✅ Loaded baseline (untrained) model")

In [ ]:
baseline_results = []
print("🚀 Starting inference on 30 clinical samples (Baseline Model)...\n")

for i, example in enumerate(test_samples):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n"
    
    inputs = tokenizer_base([prompt], return_tensors = "pt").to("cuda")
    
    outputs = model_base.generate(
        **inputs, 
        max_new_tokens = 150, 
        use_cache = True,
        temperature = 0.1,
        top_p = 0.9
    )
    
    full_response = tokenizer_base.batch_decode(outputs)[0]
    generated_text = full_response.split("### Response:\n")[-1].replace("<|end_of_text|>", "").strip()
    
    baseline_results.append({
        "ID": i + 1,
        "Clinical_Input": example['input'],
        "Baseline_Response": generated_text
    })
    
    if (i + 1) % 5 == 0:
        print(f"✅ Processed {i+1}/30 samples...")

df_baseline = pd.DataFrame(baseline_results)
df_baseline.to_csv("./baseline_model_evaluation.csv", index=False)
print("\n✅ Baseline model results saved to: ./baseline_model_evaluation.csv")

### 6.3 Comparative Analysis

Merge baseline and fine-tuned responses for side-by-side comparison.

In [ ]:
# Merge results for comparison
df_comparison = df_finetuned.merge(
    df_baseline[['ID', 'Baseline_Response']], 
    on='ID', 
    how='left'
)

# Reorder columns for clarity
df_comparison = df_comparison[[
    'ID', 
    'Instruction',
    'Clinical_Input', 
    'Ground_Truth',
    'Baseline_Response',
    'FineTuned_Response'
]]

df_comparison.to_csv("./final_model_comparison.csv", index=False)

print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print(f"✅ Total test samples evaluated: {len(df_comparison)}")
print(f"✅ Final comparison saved to: ./final_model_comparison.csv")
print("\nSample Output (First 3 rows):")
print(df_comparison.head(3).to_string(index=False))

---

## Summary

This notebook successfully completed the following pipeline:

1. **Data Processing**: Loaded 10,178 medical QA pairs from Hugging Face, filtered noise, and split into train/eval sets
2. **Hyperparameter Tuning**: Trained 9 LoRA configurations and identified the optimal setup
3. **Evaluation**: Compared fine-tuned vs. baseline model on 30 held-out clinical questions

**Output Files Generated:**
- `cleaned_medical_meadow.jsonl` - Sanitized dataset
- `train_split.jsonl` / `eval_split.jsonl` - Train/validation splits
- `grid_search_results.csv` - Hyperparameter search results
- `final_model_comparison.csv` - Side-by-side baseline vs. fine-tuned responses